In [2]:
!pip install langchain-mistralai


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install langchain langchain-community langchain-mistralai langchain-text-splitters langgraph chromadb pypdf python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import os
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_mistralai import MistralAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

load_dotenv()
llm = ChatMistralAI(model="mistral-large-latest", api_key=os.getenv("MISTRAL_API_KEY"))
print("✅ Connexion Mistral OK")

✅ Connexion Mistral OK


In [22]:
loader = PyPDFLoader("data/avene_notice.pdf")
pages = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"✅ {len(chunks)} chunks créés")

✅ 60 chunks créés


In [23]:
embeddings = MistralAIEmbeddings(api_key=os.getenv("MISTRAL_API_KEY"))
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Base vectorielle créée")

✅ Base vectorielle créée


In [24]:
class AgentState(TypedDict):
    question: str
    context: List[str]
    answer: str

def retrieve(state: AgentState):
    docs = retriever.invoke(state["question"])
    return {"context": [d.page_content for d in docs]}

def generate(state: AgentState):
    prompt = ChatPromptTemplate.from_template("""
    Tu es un expert produits Pierre Fabre / Avène.
    Réponds à la question en te basant sur le contexte fourni.
    
    Contexte: {context}
    Question: {question}
    """)
    chain = prompt | llm
    response = chain.invoke({
        "context": "\n".join(state["context"]),
        "question": state["question"]
    })
    return {"answer": response.content}

graph = StateGraph(AgentState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
agent = graph.compile()
print("✅ Agent LangGraph compilé")

✅ Agent LangGraph compilé


In [9]:
class AgentState(TypedDict):
    question: str
    context: List[str]
    answer: str

def retrieve(state: AgentState):
    docs = retriever.invoke(state["question"])
    return {"context": [d.page_content for d in docs]}

def generate(state: AgentState):
    prompt = ChatPromptTemplate.from_template("""
    Tu es un expert produits Pierre Fabre / Avène.
    Réponds à la question en te basant sur le contexte fourni.
    
    Contexte: {context}
    Question: {question}
    """)
    chain = prompt | llm
    response = chain.invoke({
        "context": "\n".join(state["context"]),
        "question": state["question"]
    })
    return {"answer": response.content}

graph = StateGraph(AgentState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
agent = graph.compile()
print("✅ Agent LangGraph compilé")

✅ Agent LangGraph compilé


In [31]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     -------------------------------------- 981.5/981.5 kB 9.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=5331b9068d6d0c02760318014b996afb3859d5442db3ecaab053657ba754af73
  Stored in directory: c:\users\jeane\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
from langdetect import detect
from langchain_core.messages import SystemMessage, HumanMessage

def retrieve(state: AgentState):
    docs = retriever.invoke(state["question"])
    return {"context": [d.page_content for d in docs]}

def generate(state: AgentState):
    # Étape 1 : détecter la langue AVANT d'appeler le LLM
    lang_code = detect(state["question"])  # "en", "fr", "es"
    lang_map = {"en": "English", "fr": "French", "es": "Spanish"}
    target_lang = lang_map.get(lang_code, "French")
    
    # Étape 2 : générer la réponse (peu importe la langue)
    response = llm.invoke([
        SystemMessage(content="Tu es un expert produits Avène / Pierre Fabre."),
        HumanMessage(content=f"Contexte:\n{chr(10).join(state['context'])}\n\nQuestion: {state['question']}")
    ])
    raw_answer = response.content
    
    # Étape 3 : traduire si nécessaire
    if target_lang != "French":
        translation = llm.invoke([
            SystemMessage(content=f"You are a professional translator. Translate the following text to {target_lang}. Output ONLY the translated text, nothing else."),
            HumanMessage(content=raw_answer)
        ])
        return {"answer": translation.content}
    
    return {"answer": raw_answer}

graph = StateGraph(AgentState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
agent = graph.compile()
print("✅ Agent multilingue compilé")

✅ Agent multilingue compilé


In [33]:
# Test anglais
result = agent.invoke({"question": "What are the indications of this Avène product?"})
print(result["answer"])

Based on the context you've shared, here are **Avène** product recommendations tailored to support someone undergoing treatment (likely for cancer, with side effects such as hair loss, skin sensitivity, etc.), along with their key benefits:

---

### **1. Care for Sensitive, Reactive, or Irritated Skin (Post-Treatment)**
#### **Avène Tolerance Control Soothing Skin Recovery Cream**
- **Key Benefits**:
  - For sensitive, reactive, or irritated skin during or after treatment (radiotherapy, chemotherapy).
  - Repairs the skin barrier, soothes redness, and relieves tightness.
  - Fragrance-free, alcohol-free, and hypoallergenic.
- **Best for**: Ideal after treatment sessions to calm irritation.

#### **Avène Cicalfate+ Restorative Protective Cream**
- **Key Benefits**:
  - For damaged, irritated skin prone to redness or micro-tears (e.g., from wig friction).
  - Accelerates healing with **copper and zinc sulfate**.
  - Protects against bacteria (thanks to **Ricinus Communis**).
- **Best fo

In [34]:
result = agent.invoke({"question": "¿Cuáles son las indicaciones de este producto Avène?"})
print(result["answer"])

Según el contexto que compartes, que parece estar relacionado con un acompañamiento compasivo (probablemente para una persona en tratamiento médico, como quimioterapia), voy a presentarte los productos **Avène** adaptados a las necesidades mencionadas (cuidados de la piel, pañuelos, uñas, etc.), así como sus **indicaciones específicas**.

---

### **1. Cuidados para pieles sensibles o debilitadas**
**Contexto**: Los tratamientos médicos (quimioterapia, radioterapia) pueden hacer que la piel se vuelva **reactiva, seca, irritada o hipersensible**.

#### **Gama Avène adaptada**:
- **Tolerancia Control Crema Calmante**:
  - **Indicaciones**: Pieles sensibles, reactivas, irritadas o propensas al enrojecimiento. Adecuada para pieles debilitadas por los tratamientos.
  - **Beneficios**: Alivia de inmediato, restaura la barrera cutánea, reduce la tirantez.
  - **Textura**: Crema ligera, sin perfume, hipoalergénica.

- **XeraCalm AD Crema Reparadora de Lípidos**:
  - **Indicaciones**: Pieles mu